In [ ]:
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

FRAME_NUMS = 50

# Add repo to path
repo_root = Path("/coc/flash7/acheluva3/EgoVerse")
sys.path.insert(0, str(repo_root))


import sys
from pathlib import Path
from typing import List, Tuple

from scipy.spatial.transform import Rotation
from tabulate import tabulate

print("✅ Libraries imported successfully")

In [ ]:
def pose_to_transform(pose: np.ndarray) -> np.ndarray:
    """
    Convert 7DOF pose to a 4x4 homogeneous transform matrix.

    Args:
        pose: Array of shape (7,) with [x, y, z, qw, qx, qy, qz] (quat in WXYZ).

    Returns:
        Array of shape (4, 4) representing SE(3) transform (rotation + translation).
    """
    x, y, z, qw, qx, qy, qz = pose
    # SciPy from_quat expects (x, y, z, w); pose[3:7] is (w, x, y, z) = WXYZ
    rotation = Rotation.from_quat([qx, qy, qz, qw])
    T = np.eye(4)
    T[:3, :3] = rotation.as_matrix()
    T[:3, 3] = [x, y, z]
    return T

def extract_camera_transforms(egomotion: np.ndarray) -> List[np.ndarray]:
    """
    Parse egomotion array into per-frame camera poses in world frame.

    Egomotion format: each row has columns [?, X, Y, Z, ?, ?, ?, qx, qy, qz, qw].
    Quaternion is XYZW (SciPy); used internally, not exposed.


    Args:
        egomotion: Array of shape (T, 11+), one row per frame.


    Returns:
        List of T 4x4 homogeneous transforms (wTc), one per frame.
    """
    transforms = []

    for i in range(len(egomotion)):
        # Translation in world frame (columns 1-3)
        t = egomotion[i, 1:4]  # X, Y, Z

        # Quaternion (columns 7-10): SciPy expects (x, y, z, w)
        q = egomotion[i, 7:11]

        # Convert quaternion -> rotation matrix
        R_mat = Rotation.from_quat(q).as_matrix()

        # Build homogeneous SE(3)
        T = np.eye(4)
        T[:3, :3] = R_mat
        T[:3, 3] = t

        transforms.append(T)

    return transforms

def transform_to_pose(T: np.ndarray) -> np.ndarray:
    """
    Convert 4x4 homogeneous transform matrix to 7DOF pose in WXYZ format.

    Args:
        T: Array of shape (4, 4) representing SE(3) transform.

    Returns:
        Array of shape (7,) with [x, y, z, qw, qx, qy, qz] (quat in WXYZ).
    """
    pos = T[:3, 3]
    rotation = Rotation.from_matrix(T[:3, :3])
    quat_xyzw = rotation.as_quat()  # SciPy returns (x, y, z, w)
    quat_wxyz = np.array([quat_xyzw[3], quat_xyzw[0], quat_xyzw[1], quat_xyzw[2]])
    return np.concatenate([pos, quat_wxyz])

In [ ]:
hands = pd.read_csv("/coc/flash7/acheluva3/EgoVerse/temp_download/hands.csv")
frames = pd.read_csv("/coc/flash7/acheluva3/EgoVerse/temp_download/frames.csv")
egomotion = np.loadtxt("/coc/flash7/acheluva3/EgoVerse/temp_download/egomotion.txt")

import logging
logger = logging.getLogger("mecka_test_processed_viz")


def compute_hand_pose_xyzquat(keypoints: np.ndarray, hand_index: int) -> np.ndarray:
    """
    Compute 7DOF pose from 21 hand keypoints in camera frame.

    Uses wrist position (keypoint 0) and constructs a right-handed frame from:
    - Forward: direction from wrist to middle finger base (keypoint 9)
    - Up: cross product of thumb (keypoint 5) and pinky (keypoint 17) directions
    - Right: cross(forward, up), then orthonormalized

    Args:
        keypoints: Array of shape (21, 3) with 3D positions of hand landmarks.

    Returns:
        Tuple of (pose_7dof, wrist_7dof), each shape (7,) with [x, y, z, qw, qx, qy, qz] (WXYZ).
        Returns (zeros, zeros) or (position+identity_quat, copy) in degenerate cases.
    """
    # Identity quaternion in WXYZ for degenerate cases
    quat_wxyz_identity = np.array([1.0, 0.0, 0.0, 0.0], dtype=np.float64)

    rot_left = np.array([[0, 0, 1], [-1, 0, 0], [0, -1, 0]])
    rot_right = np.array([[0, 0, 1], [1, 0, 0], [0, 1, 0]])

    if np.allclose(keypoints, 0):
        fallback = np.concatenate([np.zeros(3), quat_wxyz_identity])
        return fallback, fallback.copy()

    # Use wrist as origin; keypoint 0 is wrist in MediaPipe-style hand model
    centroid = np.mean(
        [keypoints[0], keypoints[17], keypoints[13], keypoints[9], keypoints[5]], axis=0
    )
    position = centroid
    wrist = keypoints[0]
    middle_base = keypoints[9]  # Base of middle finger

    # Forward axis: wrist → middle finger base
    forward = middle_base - wrist
    if np.linalg.norm(forward) < 1e-6:
        logger.warning(
            "Forward direction norm too small, returning position + identity quat"
        )
        fallback = np.concatenate([position, quat_wxyz_identity])
        return fallback, fallback.copy()
    forward = forward / np.linalg.norm(forward)

    # Up axis: cross product of thumb→wrist and pinky→wrist
    thumb_dir = keypoints[5] - wrist
    pinky_dir = keypoints[17] - wrist
    up = np.cross(thumb_dir, pinky_dir)

    if np.linalg.norm(up) < 1e-6:
        logger.warning(
            "Up direction norm too small, returning position + identity quat"
        )
        fallback = np.concatenate([position, quat_wxyz_identity])
        return fallback, fallback.copy()
    up = up / np.linalg.norm(up)

    # Right axis: cross(forward, up), then re-orthonormalize
    right = np.cross(forward, up)
    right = right / np.linalg.norm(right)
    up = np.cross(right, forward)
    right = right * -1  # Flip for right-handed frame

    rot_matrix = np.column_stack([forward, right, up])

    if hand_index == 0:
        rot_matrix = rot_matrix @ rot_left
    else:
        rot_matrix = rot_matrix @ rot_right

    quat_xyzw = Rotation.from_matrix(rot_matrix).as_quat()  # SciPy returns (x, y, z, w)
    quat_wxyz = np.array([quat_xyzw[3], quat_xyzw[0], quat_xyzw[1], quat_xyzw[2]])
    return (
        np.concatenate([position, quat_wxyz]),
        np.concatenate([wrist, quat_wxyz]),
    )


def extract_hand_data(
        hands_df: pd.DataFrame,
        frames_df: pd.DataFrame,
        arm: str,
        camera_transforms: List[np.ndarray],
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        Extract hand poses and keypoints from hands CSV and transform to world frame.


        Hands CSV has columns: frame, hand_index (0=left, 1=right), landmark_index,
        world_x, world_y, world_z. Hand poses are computed from keypoints in camera
        frame via compute_hand_pose_xyzquat, then transformed to world using wTc.


        Args:
            hands_df: DataFrame with hand landmark positions per frame.
            frames_df: DataFrame defining frame count and sync (used for shape).
            arm: Unused; kept for API compatibility.
            camera_transforms: List of wTc 4x4 transforms per frame.


        Returns:
            Tuple of (hand_poses_world, hand_keypoints_world):
                - hand_poses_world: (T, 14) [left_7dof, right_7dof] xyz+quat(WXYZ) in world frame.
                - hand_keypoints_world: (T, 2, 21, 3) [left_21kp, right_21kp] in world.
        """
        num_frames = len(frames_df)
        hand_poses = np.zeros((num_frames, 14))
        hand_keypoints = np.zeros((num_frames, 2, 21, 3))
        wrist_poses = np.zeros((num_frames, 14))

        for frame_idx in range(num_frames):
            for hand_index in [0, 1]:
                hand_data = hands_df[
                    (hands_df["frame"] == frame_idx)
                    & (hands_df["hand_index"] == hand_index)
                ].sort_values("landmark_index")

                if len(hand_data) == 21:
                    kp = hand_data[
                        ["world_x", "world_y", "world_z"]
                    ].values  # (21, 3) in camera frame
                    wTc = camera_transforms[frame_idx]

                    # Transform keypoints from camera frame to world frame (same as hand poses)
                    kp_h = np.concatenate([kp, np.ones((21, 1))], axis=1)  # (21, 4)
                    kp_world = (wTc @ kp_h.T).T[:, :3]  # (21, 3)
                    hand_keypoints[frame_idx, hand_index] = kp_world

                    # Hand pose in camera frame -> transform to world via wTc
                    pose_xyzquat, wrist_xyzquat = compute_hand_pose_xyzquat(
                        kp, hand_index
                    )

                    T_hand_cam = pose_to_transform(pose_xyzquat)
                    T_wrist_cam = pose_to_transform(wrist_xyzquat)

                    T_hand_world = wTc @ T_hand_cam
                    pose_xyzquat = transform_to_pose(T_hand_world)

                    T_wrist_world = wTc @ T_wrist_cam
                    wrist_xyzquat = transform_to_pose(T_wrist_world)

                    hand_poses[frame_idx, hand_index * 7 : (hand_index + 1) * 7] = (
                        pose_xyzquat
                    )
                    wrist_poses[frame_idx, hand_index * 7 : (hand_index + 1) * 7] = (
                        wrist_xyzquat
                    )

        return hand_poses, hand_keypoints, wrist_poses

T_cam_list = extract_camera_transforms(egomotion)
hand_poses, hand_keypoints, wrist_poses = extract_hand_data(hands, frames, "both", T_cam_list)
# Clip to the first 10 rows
poses = hand_poses[:FRAME_NUMS]
keypoints = hand_keypoints[:FRAME_NUMS]

print(tabulate(poses, headers=["lx", "ly", "lz", "lyaw", "lpitch", "lroll",
                               "rx", "ry", "rz", "ryaw", "rpitch", "rroll"],
               tablefmt="grid", floatfmt=".2f"))


In [ ]:
def extract_head_poses(egomotion: np.ndarray) -> np.ndarray:
    """
    Extract head (camera) poses from egomotion with axis remap for pipeline convention.


    Applies a basis change so output matches expected coordinate frame:
    (x, y, z) -> (-y, -z, x). Output is position + quaternion for each frame.


    Args:
        egomotion: Array of shape (T, 11+), one row per frame.


    Returns:
        Array of shape (T, 7) with [x, y, z, qw, qx, qy, qz] (quat in WXYZ) per frame.
    """
    # Frame remap: (x, y, z) -> (-y, -z, x) to match expected head pose convention
    R_fix = np.array(
        [
            [0, -1, 0],
            [0, 0, -1],
            [1, 0, 0],
        ],
        dtype=np.float64,
    )

    T_fix = np.eye(4, dtype=np.float64)
    T_fix[:3, :3] = R_fix

    # Construct inverse SE(3) transform matrix for the rotation remap
    R_fix_inv = R_fix.T
    T_fix_inv = np.eye(4, dtype=np.float64)
    T_fix_inv[:3, :3] = R_fix_inv

    adjusted_head_poses = []
    for i in range(len(egomotion)):
        # Build camera pose in world frame: wTc
        t = np.asarray(egomotion[i, 1:4], dtype=np.float64)  # world translation
        q = np.asarray(egomotion[i, 7:11], dtype=np.float64)  # quat (x, y, z, w)
        R_wc = Rotation.from_quat(q).as_matrix()

        # Build SE(3) transform matrix from translation and rotation
        T_wc = np.eye(4, dtype=np.float64)
        T_wc[:3, :3] = R_wc
        T_wc[:3, 3] = t

        # Apply axis remap: wTc' = wTc @ inv(T_fix)
        T_wc_adj = T_wc @ T_fix_inv

        # Output head pose as [x, y, z, qw, qx, qy, qz] (WXYZ)
        xyz_adj = T_wc_adj[:3, 3]
        R_wc_adj = Rotation.from_matrix(T_wc_adj[:3, :3])
        quat_xyzw = R_wc_adj.as_quat()  # SciPy returns (x, y, z, w)
        quat_wxyz = np.array(
            [quat_xyzw[3], quat_xyzw[0], quat_xyzw[1], quat_xyzw[2]]
        )
        adjusted_head_poses.append(np.concatenate([xyz_adj, quat_wxyz]))

    return np.asarray(adjusted_head_poses)

In [ ]:
# --- In-memory batch construction + transform pipeline (no Zarr required) ---

import importlib.util

import imageio_ffmpeg
import mediapy as mpy
import numpy as np
import torch

from egomimic.utils.egomimicUtils import INTRINSICS, cam_frame_to_cam_pixels, draw_actions
from scipy.spatial.transform import Rotation as R
import cv2

# Ensure mediapy can find an ffmpeg executable in this environment
mpy.set_ffmpeg(imageio_ffmpeg.get_ffmpeg_exe())

intrinsics_key = "mecka"
image_key = "images.front_1"
action_key = "actions_cartesian"

# Which "time index" to visualize from the already-loaded arrays
T0 = 0

# Action-chunk horizon (like the Zarr key_map "horizon")
ACTION_HORIZON = 30

# Match canonical ARIA bimanual pipeline settings
ACTION_CHUNK_LENGTH = 100
ACTION_STRIDE = 3

# Video source for the background image
VIDEO_PATH = "/coc/flash7/acheluva3/EgoVerse/temp_download/video.mp4"

# For MECKA, the intrinsics in egomimicUtils.py are defined for 640x360.
# Instead of scaling intrinsics, we downsample frames to match them.
TARGET_FRAME_WH = (640, 360)  # (W, H)


def _downsample_frame_rgb(frame_rgb: np.ndarray, target_wh: tuple[int, int] = TARGET_FRAME_WH) -> np.ndarray:
    if frame_rgb.ndim != 3 or frame_rgb.shape[2] != 3:
        raise ValueError(f"Expected RGB image (H,W,3), got {frame_rgb.shape}")
    tgt_w, tgt_h = target_wh
    h, w = frame_rgb.shape[:2]
    if (w, h) == (tgt_w, tgt_h):
        return frame_rgb
    return cv2.resize(frame_rgb, (tgt_w, tgt_h), interpolation=cv2.INTER_AREA)


def _split_action_pose(actions: np.ndarray):
    # 14D layout: [L xyz ypr g, R xyz ypr g]
    # 12D layout: [L xyz ypr, R xyz ypr]
    if actions.shape[-1] == 14:
        left_xyz = actions[:, :3]
        left_ypr = actions[:, 3:6]
        right_xyz = actions[:, 7:10]
        right_ypr = actions[:, 10:13]
    elif actions.shape[-1] == 12:
        left_xyz = actions[:, :3]
        left_ypr = actions[:, 3:6]
        right_xyz = actions[:, 6:9]
        right_ypr = actions[:, 9:12]
    else:
        raise ValueError(f"Unsupported action dim {actions.shape[-1]}")
    return left_xyz, left_ypr, right_xyz, right_ypr


def viz_batch(batch, image_key: str, action_key: str, intrinsics_key: str):
    img = batch[image_key][0].detach().cpu()
    if img.shape[0] in (1, 3):
        img = img.permute(1, 2, 0)
    img_np = img.numpy()

    if img_np.dtype != np.uint8:
        if img_np.max() <= 1.0:
            img_np = (img_np * 255.0).clip(0, 255).astype(np.uint8)
        else:
            img_np = img_np.clip(0, 255).astype(np.uint8)
    if img_np.shape[-1] == 1:
        img_np = np.repeat(img_np, 3, axis=-1)

    intrinsics = INTRINSICS[intrinsics_key]
    actions = batch[action_key][0].detach().cpu().numpy()
    left_xyz, _, right_xyz, _ = _split_action_pose(actions)

    vis = draw_actions(
        img_np.copy(),
        type="xyz",
        color="Blues",
        actions=left_xyz,
        extrinsics=None,
        intrinsics=intrinsics,
        arm="left",
    )
    vis = draw_actions(
        vis,
        type="xyz",
        color="Reds",
        actions=right_xyz,
        extrinsics=None,
        intrinsics=intrinsics,
        arm="right",
    )
    return vis


def viz_batch_ypr(batch, image_key: str, action_key: str, intrinsics_key: str, axis_len_m=0.04):
    img = batch[image_key][0].detach().cpu()
    if img.shape[0] in (1, 3):
        img = img.permute(1, 2, 0)
    img_np = img.numpy()

    if img_np.dtype != np.uint8:
        if img_np.max() <= 1.0:
            img_np = (img_np * 255.0).clip(0, 255).astype(np.uint8)
        else:
            img_np = img_np.clip(0, 255).astype(np.uint8)
    if img_np.shape[-1] == 1:
        img_np = np.repeat(img_np, 3, axis=-1)

    intrinsics = INTRINSICS[intrinsics_key]
    actions = batch[action_key][0].detach().cpu().numpy()
    left_xyz, left_ypr, right_xyz, right_ypr = _split_action_pose(actions)

    vis = img_np.copy()

    def _draw_axis_color_legend(frame):
        h, w = frame.shape[:2]
        x_right = w - 12
        y_start = 14
        y_step = 12
        line_len = 24
        axis_legend = [
            ("x", (255, 0, 0)),
            ("y", (0, 255, 0)),
            ("z", (0, 0, 255)),
        ]
        for i, (name, color) in enumerate(axis_legend):
            y = y_start + i * y_step
            x0 = x_right - line_len
            x1 = x_right
            cv2.line(frame, (x0, y), (x1, y), color, 3)
            cv2.putText(
                frame,
                name,
                (x0 - 12, y + 4),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.4,
                color,
                1,
                cv2.LINE_AA,
            )
        return frame

    def _draw_rotation_at_palm(frame, xyz_seq, ypr_seq, label, anchor_color):
        if len(xyz_seq) == 0 or len(ypr_seq) == 0:
            return frame

        palm_xyz = xyz_seq[0]
        palm_ypr = ypr_seq[0]
        rot = R.from_euler("ZYX", palm_ypr, degrees=False).as_matrix()

        axis_points_cam = np.vstack(
            [
                palm_xyz,
                palm_xyz + rot[:, 0] * axis_len_m,
                palm_xyz + rot[:, 1] * axis_len_m,
                palm_xyz + rot[:, 2] * axis_len_m,
            ]
        )

        px = cam_frame_to_cam_pixels(axis_points_cam, intrinsics)[:, :2]
        if not np.isfinite(px).all():
            return frame

        pts = np.round(px).astype(np.int32)

        h, w = frame.shape[:2]
        x0, y0 = pts[0]
        if not (0 <= x0 < w and 0 <= y0 < h):
            return frame

        cv2.circle(frame, (x0, y0), 4, anchor_color, -1)
        axis_colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255)]

        for i, color in enumerate(axis_colors, start=1):
            x1, y1 = pts[i]
            if 0 <= x1 < w and 0 <= y1 < h:
                cv2.line(frame, (x0, y0), (x1, y1), color, 2)
                cv2.circle(frame, (x1, y1), 2, color, -1)

        cv2.putText(
            frame,
            label,
            (x0 + 6, max(12, y0 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.4,
            anchor_color,
            1,
            cv2.LINE_AA,
        )
        return frame

    vis = _draw_rotation_at_palm(vis, left_xyz, left_ypr, "L rot", (255, 180, 80))
    vis = _draw_rotation_at_palm(vis, right_xyz, right_ypr, "R rot", (80, 180, 255))
    vis = _draw_axis_color_legend(vis)
    return vis


# --- 1) Read a single background video frame ---
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise FileNotFoundError(f"Could not open VIDEO_PATH={VIDEO_PATH}")
cap.set(cv2.CAP_PROP_POS_FRAMES, float(T0))
ret, frame_bgr = cap.read()
cap.release()
if not ret:
    raise RuntimeError(f"Failed to read frame {T0} from {VIDEO_PATH}")
frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
frame_rgb = _downsample_frame_rgb(frame_rgb)
img_t = torch.from_numpy(frame_rgb).permute(2, 0, 1).contiguous().float() / 255.0


# --- 2) Construct an in-memory "episode item" dict (like process_episode / key_map) ---
# Use data computed in previous cells:
#   - hand_poses: (T, 14) [left_7dof, right_7dof] xyz + quat(WXYZ) in world frame
#   - egomotion:  (T, ...) camera motion
head_poses = extract_head_poses(egomotion)  # (T, 7) xyz + quat(WXYZ)

T_total = int(hand_poses.shape[0])
if not (0 <= T0 < T_total):
    raise ValueError(f"T0={T0} out of range for T_total={T_total}")

horizon = int(min(ACTION_HORIZON, T_total - T0))
if horizon < 2:
    raise ValueError(f"Not enough frames for horizon={horizon} at T0={T0}")

left_world = np.asarray(hand_poses[:, 0:7], dtype=np.float64)
right_world = np.asarray(hand_poses[:, 7:14], dtype=np.float64)

item_np = {
    # Target frame pose (single): shape (7,)
    "obs_head_pose": np.asarray(head_poses[T0], dtype=np.float64),
    # Observations (single): shape (7,)
    "left.obs_ee_pose": np.asarray(left_world[T0], dtype=np.float64),
    "right.obs_ee_pose": np.asarray(right_world[T0], dtype=np.float64),
    # "Action" chunks (sequence): shape (H, 7)
    "left.action_ee_pose": np.asarray(left_world[T0 : T0 + horizon], dtype=np.float64),
    "right.action_ee_pose": np.asarray(right_world[T0 : T0 + horizon], dtype=np.float64),
}


# --- 3) Load the transform module WITHOUT importing egomimic.rldb.zarr/__init__.py ---
# This avoids any unrelated package import side-effects; we only need the transform definitions.
action_chunk_path = repo_root / "egomimic/rldb/zarr/action_chunk_transforms.py"
spec = importlib.util.spec_from_file_location("action_chunk_transforms_noinit", action_chunk_path)
if spec is None or spec.loader is None:
    raise RuntimeError(f"Failed to load module spec for {action_chunk_path}")
_action_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(_action_mod)

transform_list = _action_mod.build_aria_bimanual_transform_list(
    chunk_length=ACTION_CHUNK_LENGTH,
    stride=ACTION_STRIDE,
    target_world="obs_head_pose",
    target_world_is_quat=True,
)

def _actions_cartesian_at(t0: int) -> np.ndarray:
    """Return actions_cartesian (chunk_length, 12) for a given frame index t0."""
    t0 = int(t0)
    T_total = int(hand_poses.shape[0])
    if not (0 <= t0 < T_total):
        raise ValueError(f"t0={t0} out of range for T_total={T_total}")

    horizon = int(min(ACTION_HORIZON, T_total - t0))
    if horizon < 2:
        raise ValueError(f"Not enough frames for horizon={horizon} at t0={t0}")

    head_poses_local = extract_head_poses(egomotion)  # (T, 7) xyz + quat(WXYZ)
    left_world = np.asarray(hand_poses[:, 0:7], dtype=np.float64)
    right_world = np.asarray(hand_poses[:, 7:14], dtype=np.float64)

    item_np = {
        "obs_head_pose": np.asarray(head_poses_local[t0], dtype=np.float64),
        "left.obs_ee_pose": np.asarray(left_world[t0], dtype=np.float64),
        "right.obs_ee_pose": np.asarray(right_world[t0], dtype=np.float64),
        "left.action_ee_pose": np.asarray(left_world[t0 : t0 + horizon], dtype=np.float64),
        "right.action_ee_pose": np.asarray(right_world[t0 : t0 + horizon], dtype=np.float64),
    }

    out_local = dict(item_np)
    for tr in transform_list:
        out_local = tr.transform(out_local)

    return np.asarray(out_local[action_key], dtype=np.float32)


def viz_batch_xyz_and_rot(batch, image_key: str, action_key: str, intrinsics_key: str, axis_len_m=0.04):
    """Overlay BOTH XYZ dots and rotation axes on the same frame."""
    img = batch[image_key][0].detach().cpu()
    if img.shape[0] in (1, 3):
        img = img.permute(1, 2, 0)
    img_np = img.numpy()

    if img_np.dtype != np.uint8:
        if img_np.max() <= 1.0:
            img_np = (img_np * 255.0).clip(0, 255).astype(np.uint8)
        else:
            img_np = img_np.clip(0, 255).astype(np.uint8)
    if img_np.shape[-1] == 1:
        img_np = np.repeat(img_np, 3, axis=-1)

    intrinsics = INTRINSICS[intrinsics_key]
    actions = batch[action_key][0].detach().cpu().numpy()
    left_xyz, left_ypr, right_xyz, right_ypr = _split_action_pose(actions)

    vis = draw_actions(
        img_np.copy(),
        type="xyz",
        color="Blues",
        actions=left_xyz,
        extrinsics=None,
        intrinsics=intrinsics,
        arm="left",
    )
    vis = draw_actions(
        vis,
        type="xyz",
        color="Reds",
        actions=right_xyz,
        extrinsics=None,
        intrinsics=intrinsics,
        arm="right",
    )

    def _draw_axis_color_legend(frame):
        h, w = frame.shape[:2]
        x_right = w - 12
        y_start = 14
        y_step = 12
        line_len = 24
        axis_legend = [
            ("x", (255, 0, 0)),
            ("y", (0, 255, 0)),
            ("z", (0, 0, 255)),
        ]
        for i, (name, color) in enumerate(axis_legend):
            y = y_start + i * y_step
            x0 = x_right - line_len
            x1 = x_right
            cv2.line(frame, (x0, y), (x1, y), color, 3)
            cv2.putText(
                frame,
                name,
                (x0 - 12, y + 4),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.4,
                color,
                1,
                cv2.LINE_AA,
            )
        return frame

    def _draw_rotation_at_palm(frame, xyz_seq, ypr_seq, label, anchor_color):
        if len(xyz_seq) == 0 or len(ypr_seq) == 0:
            return frame

        palm_xyz = xyz_seq[0]
        palm_ypr = ypr_seq[0]
        rot = R.from_euler("ZYX", palm_ypr, degrees=False).as_matrix()

        axis_points_cam = np.vstack(
            [
                palm_xyz,
                palm_xyz + rot[:, 0] * axis_len_m,
                palm_xyz + rot[:, 1] * axis_len_m,
                palm_xyz + rot[:, 2] * axis_len_m,
            ]
        )

        px = cam_frame_to_cam_pixels(axis_points_cam, intrinsics)[:, :2]
        if not np.isfinite(px).all():
            return frame

        pts = np.round(px).astype(np.int32)

        h, w = frame.shape[:2]
        x0, y0 = pts[0]
        if not (0 <= x0 < w and 0 <= y0 < h):
            return frame

        cv2.circle(frame, (x0, y0), 4, anchor_color, -1)
        axis_colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255)]

        for i, color in enumerate(axis_colors, start=1):
            x1, y1 = pts[i]
            if 0 <= x1 < w and 0 <= y1 < h:
                cv2.line(frame, (x0, y0), (x1, y1), color, 2)
                cv2.circle(frame, (x1, y1), 2, color, -1)

        cv2.putText(
            frame,
            label,
            (x0 + 6, max(12, y0 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.4,
            anchor_color,
            1,
            cv2.LINE_AA,
        )
        return frame

    vis = _draw_rotation_at_palm(vis, left_xyz, left_ypr, "L rot", (255, 180, 80))
    vis = _draw_rotation_at_palm(vis, right_xyz, right_ypr, "R rot", (80, 180, 255))
    vis = _draw_axis_color_legend(vis)
    return vis


def _overlay_image_for_frame(frame_rgb_uint8: np.ndarray, t0: int) -> np.ndarray:
    frame_rgb_uint8 = _downsample_frame_rgb(frame_rgb_uint8)
    img_t_local = (
        torch.from_numpy(frame_rgb_uint8)
        .permute(2, 0, 1)
        .contiguous()
        .float()
        / 255.0
    )
    actions_np_local = _actions_cartesian_at(t0)
    batch_local = {
        image_key: img_t_local[None, ...],
        action_key: torch.from_numpy(actions_np_local)[None, ...],
    }
    return viz_batch_xyz_and_rot(
        batch_local,
        image_key=image_key,
        action_key=action_key,
        intrinsics_key=intrinsics_key,
    )


# Sanity-check single frame visualization
out_actions = _actions_cartesian_at(T0)
print(f"actions_cartesian@T0: shape={out_actions.shape} dtype={out_actions.dtype}")

vis = viz_batch(
    {image_key: img_t[None, ...], action_key: torch.from_numpy(out_actions)[None, ...]},
    image_key=image_key,
    action_key=action_key,
    intrinsics_key=intrinsics_key,
)
mpy.show_image(vis)

vis_all = _overlay_image_for_frame(frame_rgb, T0)
mpy.show_image(vis_all)


In [ ]:
# --- Render an overlay video ---

START_FRAME = 0
NUM_FRAMES = 120
FRAME_STRIDE = 1
FPS = 30

OUT_PATH = "mecka_overlay.mp4"

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise FileNotFoundError(f"Could not open VIDEO_PATH={VIDEO_PATH}")

cap.set(cv2.CAP_PROP_POS_FRAMES, float(START_FRAME))

overlay_frames = []

for i in range(NUM_FRAMES):
    t = START_FRAME + i * FRAME_STRIDE

    ret, frame_bgr = cap.read()
    if not ret:
        print(f"Stopped early at i={i} (t={t})")
        break

    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    frame_rgb = _downsample_frame_rgb(frame_rgb)

    # If you stride, skip frames in the capture stream too
    if FRAME_STRIDE > 1:
        for _ in range(FRAME_STRIDE - 1):
            _ = cap.grab()

    try:
        overlay = _overlay_image_for_frame(frame_rgb, t)
    except Exception as e:
        print(f"Overlay failed at t={t}: {e}")
        overlay = frame_rgb

    overlay_frames.append(overlay)

cap.release()

print(f"Writing video: {OUT_PATH} with {len(overlay_frames)} frames")
mpy.write_video(OUT_PATH, overlay_frames, fps=FPS)

# Show the rendered video inline
mpy.show_video(overlay_frames, fps=FPS)
